<a href="https://colab.research.google.com/github/llazdll/Spark_BookRating/blob/main/BookRating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, format_number, concat, substring, to_timestamp, date_format,count



In [29]:

spark=SparkSession.builder.appName("Book Rating").master("local[*]").getOrCreate()


In [30]:
!wget -O Book-Ratings.csv https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Book-Ratings.csv
!wget -O Books.csv https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Books.csv
!wget -O Users.csv https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Users.csv

--2026-08-18 12:08:37--  https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Book-Ratings.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30682266 (29M) [application/octet-stream]
Saving to: ‘Book-Ratings.csv’

Book-Ratings.csv    100%[===================>]  29.26M  92.8MB/s    in 0.3s    

2026-08-18 12:08:38 (92.8 MB/s) - ‘Book-Ratings.csv’ saved [30682266/30682266]

--2026-08-18 12:08:38--  https://raw.githubusercontent.com/llazdll/Spark_BookRating/main/BooksRating-CSV/Books.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, await

In [31]:

df_BookRating= spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv("Book-Ratings.csv")


In [32]:
df_BookRating.show()
df_BookRating.printSchema()

+------+----------+----+
|userid|      isbn|rate|
+------+----------+----+
|276725|034545104X|   0|
|276726|0155061224|   5|
|276727|0446520802|   0|
|276729|052165615X|   3|
|276729|0521795028|   6|
|276733|2080674722|   0|
|276736|3257224281|   8|
|276737|0600570967|   6|
|276744|038550120X|   7|
|276745| 342310538|  10|
|276746|0425115801|   0|
|276746|0449006522|   0|
|276746|0553561618|   0|
|276746|055356451X|   0|
|276746|0786013990|   0|
|276746|0786014512|   0|
|276747|0060517794|   9|
|276747|0451192001|   0|
|276747|0609801279|   0|
|276747|0671537458|   9|
+------+----------+----+
only showing top 20 rows
root
 |-- userid: integer (nullable = true)
 |-- isbn: string (nullable = true)
 |-- rate: integer (nullable = true)



In [33]:
df_Books= spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv("Books.csv")

In [34]:
df_Books.show()
df_Books.printSchema()

+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|      ISBN|           BookTitle|          BookAuthor|YearOfPublication|           Publisher|           ImageURLS|           ImageURLM|           ImageURLL|
+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|0195153448| Classical Mythology|  Mark P. O. Morford|             2002|Oxford University...|http://images.ama...|http://images.ama...|http://images.ama...|
|0002005018|        Clara Callan|Richard Bruce Wright|             2001|HarperFlamingo Ca...|http://images.ama...|http://images.ama...|http://images.ama...|
|0060973129|Decision in Normandy|        Carlo D'Este|             1991|     HarperPerennial|http://images.ama...|http://images.ama...|http://images.ama...|
|0374157065|Flu: The Story of...|    Gina Bari Kolata|    

In [35]:
df_Users= spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .csv("Users.csv")

In [36]:
df_Users.show()
df_Users.printSchema()

+------+-----------+--------------------+----+
|UserID|   USERNAME|            Location| Age|
+------+-----------+--------------------+----+
|     1|bzsufoRTLN2|  nyc, new york, usa|NULL|
|     2|fq7kfHg4VEI|stockton, califor...|  18|
|     3|W0Hbkd3xR8v|moscow, yukon ter...|NULL|
|     4|W51GahAx5Ap|porto, v.n.gaia, ...|  17|
|     5|VKN3PQ18GgN|farnborough, hant...|NULL|
|     6|h9BSgQZ5wOk|santa monica, cal...|  61|
|     7|7rdddpZpjWp| washington, dc, usa|NULL|
|     8|qiOJebWJS2i|timmins, ontario,...|NULL|
|     9|gkcxQJLS13A|germantown, tenne...|NULL|
|    10|BANPptNSbPy|albacete, wiscons...|  26|
|    11|X0ELjsCzJt0|melbourne, victor...|  14|
|    12|z4qLRN05PYT|fort bragg, calif...|NULL|
|    13|4YgtGJggfAx|barcelona, barcel...|  26|
|    14|FZW1MG3zLpo|mediapolis, iowa,...|NULL|
|    15|FHo7m8eHkPb|calgary, alberta,...|NULL|
|    16|HwLFQdzk30o|albuquerque, new ...|NULL|
|    17|dk15dEDpgbc|chesapeake, virgi...|NULL|
|    18|aNFnRPqjGL1|rio de janeiro, r...|  25|
|    19|hNw8C